In [ ]:
from __future__ import annotations

import argparse
import html
import json
import re
import sys
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any
from urllib.error import HTTPError, URLError
from urllib.parse import urljoin
from urllib.request import Request, urlopen


BASE_URL = "https://ev-database.org/"
CATALOG_URL = BASE_URL
USER_AGENT = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) EVDatabaseScraper/1.0"
REQUEST_TIMEOUT = 30
REQUEST_RETRIES = 5
REQUEST_BACKOFF_SECONDS = 2.0
CAR_CARD_TOKEN = '<div class="list-item" data-jplist-item>'


def fetch_html(url: str) -> str:
    request = Request(
        url,
        headers={
            "User-Agent": USER_AGENT,
            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
            "Accept-Language": "en-US,en;q=0.9",
        },
    )

    last_error: Exception | None = None
    for attempt in range(REQUEST_RETRIES):
        try:
            with urlopen(request, timeout=REQUEST_TIMEOUT) as response:
                return response.read().decode("utf-8", "replace")
        except HTTPError as error:
            last_error = error
            retryable = error.code in {429, 500, 502, 503, 504}
            if not retryable or attempt == REQUEST_RETRIES - 1:
                raise
        except URLError as error:
            last_error = error
            if attempt == REQUEST_RETRIES - 1:
                raise

        time.sleep(REQUEST_BACKOFF_SECONDS * (attempt + 1))

    if last_error is not None:
        raise last_error

    raise RuntimeError(f"Unable to fetch {url}")


def strip_tags(value: str | None) -> str:
    if value is None:
        return ""
    value = html.unescape(value)
    value = re.sub(r"<[^>]+>", " ", value)
    value = re.sub(r"\s+", " ", value)
    return value.strip()


def extract_first(pattern: str, text: str, flags: int = re.S) -> str:
    match = re.search(pattern, text, flags)
    return match.group(1).strip() if match else ""


def parse_number(text: str | None) -> float | None:
    if not text:
        return None
    match = re.search(r"-?\d+(?:\.\d+)?", text.replace(",", ""))
    return float(match.group(0)) if match else None


def parse_int(text: str | None) -> int | None:
    number = parse_number(text)
    return int(number) if number is not None else None


def normalize_key(value: str) -> str:
    value = strip_tags(value)
    value = value.lower()
    value = re.sub(r"[^a-z0-9]+", "_", value)
    return value.strip("_")


def parse_title(title_html: str) -> dict[str, str]:
    brand = extract_first(r'<span class="[^"]+">([^<]+)</span>', title_html)
    model = extract_first(r'<span class="model">(.*?)</span>', title_html)
    canonical = extract_first(r'<span class="hidden">([^<]+)</span>', title_html)
    display_name = strip_tags(title_html)

    if not canonical:
        canonical = display_name

    return {
        "brand": brand,
        "model": model,
        "canonical_name": canonical,
        "display_name": display_name,
    }


def parse_card(card_html: str) -> dict[str, Any]:
    href = extract_first(r'<a href="([^"]+)"\s+class="title">', card_html)
    title_html = extract_first(r'<a href="[^"]+"\s+class="title">(.*?)</a>', card_html)
    title = parse_title(title_html)

    car_id_match = re.search(r"/car/(\d+)/", href)
    car_id = int(car_id_match.group(1)) if car_id_match else None

    summary = {
        "range_km": parse_number(extract_first(r'<span class="erange_real">([^<]+)</span>', card_html)),
        "efficiency_wh_per_km": parse_number(extract_first(r'<span class="efficiency">([^<]+)</span>', card_html)),
        "weight_kg": parse_int(extract_first(r'<span class="weight_p">([^<]+)</span>', card_html)),
        "acceleration_sec": parse_number(extract_first(r'<span class="acceleration_p">([^<]+)</span>', card_html)),
        "one_stop_range_km": parse_number(extract_first(r'<span class="long_distance_total">([^<]+)</span>', card_html)),
        "battery_kwh": parse_number(extract_first(r'<span class="battery_p">([^<]+)</span>', card_html)),
        "fastcharge_kw": parse_number(extract_first(r'<span class="fastcharge_speed_print">([^<]+)</span>', card_html)),
        "towing_kg": parse_int(extract_first(r'<span class="towweight_p">([^<]+)</span>', card_html)),
        "cargo_volume_l": parse_int(extract_first(r'<span class="cargo">([^<]+)</span>', card_html)),
        "price_per_range_eur_per_km": parse_number(extract_first(r'<span class="priceperrange_p">([^<]+)</span>', card_html)),
    }

    availability = extract_first(r'<div class="availability[^>]*">([^<]+)</div>', card_html)
    drive_type = extract_first(r'data-tooltip="([^"]+Wheel Drive)"', card_html)
    segment_letter = extract_first(r'data-tooltip="Market Segment"\s+class="size-[^"]+">([A-Z])</span>', card_html)
    seats = parse_int(extract_first(r'data-tooltip="Number of seats"[^>]*>\s*(?:<i[^>]*></i>\s*)?<span>(\d+)</span>', card_html))

    prices = {
        "de": extract_first(r'<span class="country_de"[^>]*>([^<]+)</span>', card_html),
        "nl": extract_first(r'<span class="country_nl"[^>]*>([^<]+)</span>', card_html),
        "uk": extract_first(r'<span class="country_uk"[^>]*>([^<]+)</span>', card_html),
    }
    prices = {code: value for code, value in prices.items() if value}

    return {
        "car_id": car_id,
        "url": urljoin(BASE_URL, href),
        **title,
        "availability": availability,
        "drive_type": drive_type,
        "segment_letter": segment_letter,
        "seats": seats,
        "summary": summary,
        "prices": prices,
    }


def parse_section_tables(html_text: str) -> dict[str, dict[str, str]]:
    sections: dict[str, dict[str, str]] = {}
    heading_matches = list(re.finditer(r"<h2>(.*?)</h2>", html_text, re.S))

    for index, heading_match in enumerate(heading_matches):
        section_name = normalize_key(heading_match.group(1))
        section_start = heading_match.end()
        section_end = heading_matches[index + 1].start() if index + 1 < len(heading_matches) else len(html_text)
        section_html = html_text[section_start:section_end]

        rows: dict[str, str] = {}
        for row_html in re.findall(r"<tr>(.*?)</tr>", section_html, re.S):
            cells = [strip_tags(cell) for cell in re.findall(r"<t[dh]>(.*?)</t[dh]>", row_html, re.S)]
            if len(cells) == 2:
                key, value = cells
                rows[key] = value

        if rows:
            sections[section_name] = rows

    return sections


def parse_detail_page(detail_html: str) -> dict[str, Any]:
    title = strip_tags(extract_first(r"<h1[^>]*>(.*?)</h1>", detail_html))
    sections = parse_section_tables(detail_html)

    misc = sections.get("miscellaneous", {})
    battery = sections.get("battery", {})
    performance = sections.get("performance", {})
    dimensions = sections.get("dimensions_and_weight", {})
    energy = sections.get("energy_consumption", {})

    def section_value(section: dict[str, str], *labels: str) -> str:
        for label in labels:
            if label in section:
                return section[label]
        return ""

    return {
        "page_title": title,
        "body_style": section_value(misc, "Car Body"),
        "segment": section_value(misc, "Segment"),
        "seat_count": parse_int(section_value(misc, "Seats")),
        "platform": section_value(misc, "Platform"),
        "ev_dedicated_platform": section_value(misc, "EV Dedicated Platform"),
        "roof_rails": section_value(misc, "Roof Rails"),
        "heat_pump": section_value(misc, "Heat pump (HP)"),
        "hp_standard_equipment": section_value(misc, "HP Standard Equipment"),
        "battery": {
            "nominal_capacity_kwh": parse_number(section_value(battery, "Nominal Capacity *", "Nominal Capacity*", "Nominal Capacity")),
            "usable_capacity_kwh": parse_number(section_value(battery, "Useable Capacity*", "Useable Capacity *", "Useable Capacity")),
            "battery_type": section_value(battery, "Battery Type"),
            "architecture_v": parse_number(section_value(battery, "Architecture")),
            "warranty_period": section_value(battery, "Warranty Period"),
            "warranty_mileage_km": parse_number(section_value(battery, "Warranty Mileage")),
            "cathode_material": section_value(battery, "Cathode Material"),
            "pack_configuration": section_value(battery, "Pack Configuration"),
            "nominal_voltage": section_value(battery, "Nominal Voltage"),
            "form_factor": section_value(battery, "Form Factor"),
        },
        "performance": {
            "acceleration_0_100_sec": parse_number(section_value(performance, "Acceleration 0 - 100 km/h")),
            "top_speed_kmh": parse_number(section_value(performance, "Top Speed")),
            "electric_range_km": parse_number(section_value(performance, "Electric Range *", "Electric Range")),
            "total_power_kw": parse_number(section_value(performance, "Total Power")),
            "total_torque_nm": parse_number(section_value(performance, "Total Torque")),
            "drive": section_value(performance, "Drive"),
        },
        "energy_consumption": {
            "evdb_real_range_km": parse_number(section_value(energy, "Range *", "Range")),
            "vehicle_consumption_wh_per_km": parse_number(section_value(energy, "Vehicle Consumption *", "Vehicle Consumption")),
            "co2_emissions_g_per_km": parse_number(section_value(energy, "CO2 Emissions")),
            "vehicle_fuel_equivalent_l_per_100km": section_value(energy, "Vehicle Fuel Equivalent *", "Vehicle Fuel Equivalent"),
        },
        "dimensions_and_weight": {
            "length_mm": parse_number(section_value(dimensions, "Length")),
            "width_mm": parse_number(section_value(dimensions, "Width")),
            "width_with_mirrors_mm": parse_number(section_value(dimensions, "Width with mirrors")),
            "height_mm": parse_number(section_value(dimensions, "Height")),
            "wheelbase_mm": parse_number(section_value(dimensions, "Wheelbase")),
            "weight_unladen_eu_kg": parse_number(section_value(dimensions, "Weight Unladen (EU)")),
            "gross_vehicle_weight_kg": parse_number(section_value(dimensions, "Gross Vehicle Weight (GVWR)")),
            "max_payload_kg": parse_number(section_value(dimensions, "Max. Payload")),
            "cargo_volume_l": parse_number(section_value(dimensions, "Cargo Volume")),
            "cargo_volume_max_l": parse_number(section_value(dimensions, "Cargo Volume Max")),
            "cargo_volume_frunk_l": parse_number(section_value(dimensions, "Cargo Volume Frunk")),
            "roof_load_kg": parse_number(section_value(dimensions, "Roof Load")),
            "tow_hitch_possible": section_value(dimensions, "Tow Hitch Possible"),
            "towing_weight_unbraked_kg": parse_number(section_value(dimensions, "Towing Weight Unbraked")),
            "towing_weight_braked_kg": parse_number(section_value(dimensions, "Towing Weight Braked")),
            "vertical_load_max_kg": parse_number(section_value(dimensions, "Vertical Load Max")),
        },
        "raw_sections": sections,
    }


def build_catalog_dataset(limit: int | None = None) -> list[dict[str, Any]]:
    catalog_html = fetch_html(CATALOG_URL)
    cards = []

    for card_html in catalog_html.split(CAR_CARD_TOKEN)[1:]:
        card = parse_card(CAR_CARD_TOKEN + card_html)
        if card.get("car_id") is not None:
            cards.append(card)

    if limit is not None:
        cards = cards[:limit]

    vehicles: list[dict[str, Any]] = []
    for card in cards:
        vehicles.append(
            {
                "car_id": card["car_id"],
                "source_url": card["url"],
                "brand": card["brand"],
                "model": card["model"],
                "canonical_name": card["canonical_name"],
                "display_name": card["display_name"],
                "availability": card["availability"],
                "drive_type": card["drive_type"],
                "segment_letter": card["segment_letter"],
                "seats": card["seats"],
                "summary": card["summary"],
                "prices": card["prices"],
            }
        )

    return vehicles


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")


output_path = Path.cwd() / "project_ev" / "ev_database_vehicles.json"
vehicles = build_catalog_dataset()
payload = {
    "source": BASE_URL,
    "scraped_at": datetime.now(timezone.utc).isoformat(),
    "count": len(vehicles),
    "vehicles": vehicles,
}
write_json(output_path, payload)

print(f"Wrote {len(vehicles)} vehicles to {output_path}")

ImportError: cannot import name 'build_catalog_dataset' from 'ev_database_scraper' (c:\Users\tomde\OneDrive\Documentatie - professioneel - opleiding\AI pro 2025-26\Project - Gen AI\project_ev\ev_database_scraper.py)